In [1]:
import os, json
from concurrent.futures import ThreadPoolExecutor
import requests
from PIL import Image
from google.colab import drive
drive.mount("/content/drive")

REL = "/content/drive/MyDrive/coco2017_rag_preprocessing"
src = {"kb_10k": "train2017", "val_100": "train2017", "test_100": "val2017", "test_1000": "val2017"}
jobs = {}
for n, folder in src.items():
    for x in json.load(open(f"{REL}/splits/{n}.json")):
        jobs[(folder, x["file_name"])] = n
for f in ["train2017", "val2017"]:
    os.makedirs(f"{REL}/images/{f}", exist_ok=True)

def check(key):
    folder, fname = key
    path = f"{REL}/images/{folder}/{fname}"
    try:
        if not os.path.exists(path):
            r = requests.get(f"http://images.cocodataset.org/{folder}/{fname}", timeout=30)
            r.raise_for_status()
            open(path, "wb").write(r.content)
        with Image.open(path) as im: im.verify()                      # header integrity
        with Image.open(path) as im: im.load(); return key, "ok", im.mode   # full decode
    except Exception as e:
        return key, f"error: {type(e).__name__}", None

with ThreadPoolExecutor(16) as ex:
    res = list(ex.map(check, jobs))

bad = [list(k) for k, s, _ in res if s != "ok"]
report = {"expected_images": len(jobs), "valid": len(jobs) - len(bad),
          "missing_or_corrupted": bad,
          "non_rgb_images": sum(m != "RGB" for _, s, m in res if s == "ok"),
          "note": "grayscale/CMYK images are converted to RGB at embedding time (EXIF transpose + convert('RGB'))"}
os.makedirs(f"{REL}/audit_eda/audit_logs", exist_ok=True)
json.dump(report, open(f"{REL}/audit_eda/audit_logs/image_validation.json", "w"), indent=1)
print(json.dumps(report, indent=1))

Mounted at /content/drive
{
 "expected_images": 11200,
 "valid": 11200,
 "missing_or_corrupted": [],
 "non_rgb_images": 21,
 "note": "grayscale/CMYK images are converted to RGB at embedding time (EXIF transpose + convert('RGB'))"
}
